# 4.26 — Association rule mining
Association rule mining turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise. Counting itemsets becomes probabilistic evidence for rules that appear more often than chance would suggest. Save a copy to Drive to edit.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import SpectralBiclustering
from sklearn.datasets import load_digits, load_iris, make_blobs, make_moons
from sklearn.ensemble import IsolationForest
from sklearn.metrics import adjusted_rand_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KernelDensity, LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler, StandardScaler


def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty. Returns [(name, X, y_true, k), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs


def matrix_for_biclustering(X):
    X = np.asarray(X, dtype=float)
    return MinMaxScaler().fit_transform(X) + 0.001


def project_for_plot(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 2:
        return X
    centered = StandardScaler().fit_transform(X)
    u, s, vt = np.linalg.svd(centered, full_matrices=False)
    return centered @ vt[:2].T


def preview_rungs(rungs):
    for name, X, y, k in rungs:
        counts = dict(zip(*np.unique(y, return_counts=True)))
        print(name, "shape=", X.shape, "k=", k, "labels=", counts)
        print(np.round(X[:3, : min(5, X.shape[1])], 3))


def make_anomaly_ladder():
    rungs = []

    X1 = np.array([[0.0, 0.0], [0.2, 0.0], [0.0, 0.1], [4.0, 4.0]])
    y1 = np.array([0, 0, 0, 1])
    rungs.append(("D1 hand normals + one outlier", X1, y1, 2))

    for idx, (name, X, labels, k) in enumerate(cluster_ladder()[1:], start=2):
        rng = np.random.default_rng(240 + idx)
        X = np.asarray(X, dtype=float)
        n_outliers = max(8, int(0.08 * len(X)))
        lo = X.min(axis=0)
        hi = X.max(axis=0)
        span = np.maximum(hi - lo, 1.0)
        low_outliers = lo - rng.uniform(1.5, 2.5, size=(n_outliers // 2, X.shape[1])) * span
        high_outliers = hi + rng.uniform(1.5, 2.5, size=(n_outliers - n_outliers // 2, X.shape[1])) * span
        outliers = np.vstack([low_outliers, high_outliers])
        X_aug = np.vstack([X, outliers])
        y_aug = np.concatenate([np.zeros(len(X), dtype=int), np.ones(n_outliers, dtype=int)])
        rungs.append((name.replace("clusters", "normal groups") + " + synthetic outliers", X_aug, y_aug, 2))

    return rungs


def make_transaction_ladder():
    ladders = []
    base = [
        {"bread", "milk"},
        {"bread", "milk", "eggs"},
        {"bread", "eggs"},
        {"milk", "eggs"},
    ]
    labels = np.array([0, 0, 1, 1])
    ladders.append(("D1 four baskets", base, labels, 2))

    for name, X, y, k in cluster_ladder()[1:]:
        X = StandardScaler().fit_transform(X)
        transactions = []
        for row in X:
            items = set()
            for j, value in enumerate(row[: min(8, X.shape[1])]):
                if value <= -0.5:
                    items.add(f"f{j}_low")
                elif value >= 0.5:
                    items.add(f"f{j}_high")
                else:
                    items.add(f"f{j}_mid")
            transactions.append(items)
        ladders.append((name + " discretized transactions", transactions, y, k))

    return ladders


def print_table(rows, metric_name):
    print(f"{'rung':<38} {metric_name:>12}")
    for name, value in rows:
        print(f"{name:<38} {value:>12.3f}")


## The concept, built once (D1): support and confidence
The lesson's rule metrics are $$support(A\Rightarrow B)=P(A\cap B),\quad confidence=P(B|A),\quad lift=\frac{P(B|A)}{P(B)}.$$ We count itemsets directly so the arithmetic is auditable.

In [ ]:

def support(transactions, itemset):
    itemset = set(itemset)
    count = sum(1 for transaction in transactions if itemset.issubset(transaction))
    return count / len(transactions)


def rule_metrics(transactions, antecedent, consequent):
    antecedent = set(antecedent)
    consequent = set(consequent)
    both = antecedent | consequent
    support_both = support(transactions, both)
    support_a = support(transactions, antecedent)
    support_b = support(transactions, consequent)
    confidence = support_both / support_a if support_a else 0.0
    lift = confidence / support_b if support_b else 0.0
    return support_both, confidence, lift

lesson_transactions = [
    {"bread", "milk"},
    {"bread", "milk", "eggs"},
    {"bread", "eggs"},
    {"milk", "eggs"},
]
lesson_support, lesson_confidence, lesson_lift = rule_metrics(lesson_transactions, {"bread"}, {"milk"})
print(round(lesson_support, 3), round(lesson_confidence, 3), round(lesson_lift, 3))
assert round(lesson_support, 3) == 0.500


## The concept, built once (D1): Apriori-style rule search
The reusable method implements a real Apriori-style pass for one-item antecedents and consequents: frequent items are found, candidate rules are scored, then support, confidence, and lift are returned.

In [ ]:

def method(transactions, min_support=0.1, min_confidence=0.0):
    items = sorted(set().union(*transactions))
    frequent_items = []
    for item in items:
        item_support = support(transactions, {item})
        if item_support >= min_support:
            frequent_items.append(item)
    rules = []
    for antecedent in frequent_items:
        for consequent in frequent_items:
            if antecedent == consequent:
                continue
            s, c, l = rule_metrics(transactions, {antecedent}, {consequent})
            if s >= min_support and c >= min_confidence:
                rules.append({"antecedent": antecedent, "consequent": consequent, "support": s, "confidence": c, "lift": l})
    rules.sort(key=lambda rule: (rule["lift"], rule["confidence"], rule["support"]), reverse=True)
    return rules

print("lesson confidence", round(lesson_confidence, 3))
print("lesson lift", round(lesson_lift, 3))
assert round(lesson_confidence, 3) == 0.667
assert round(lesson_lift, 3) == 0.889


## The dataset ladder
D1 is a four-basket lesson dataset. D2–D5 discretize the F2 numeric ladder into transactions of increasing size and dimension.

In [ ]:

rungs = make_transaction_ladder()
for name, transactions, labels, k in rungs:
    print(name, "transactions=", len(transactions), "items in first basket=", sorted(list(transactions[0]))[:8])


## Run the same method across D1–D5
The metric is the strongest lift among rules that pass support. We also keep support and confidence so lift is never read without the underlying counts.

In [ ]:

rows = []
artifacts = []
for name, transactions, labels, k in rungs:
    min_support = max(0.05, 3 / len(transactions))
    rules = method(transactions, min_support=min_support, min_confidence=0.1)
    best = rules[0] if rules else {"support": 0.0, "confidence": 0.0, "lift": 0.0, "antecedent": "none", "consequent": "none"}
    rows.append((name, best["lift"]))
    artifacts.append((name, transactions, labels, rules[:8], best))
print_table(rows, "best lift")
for name, transactions, labels, rules, best in artifacts:
    print(name, "top rule=", best)


## Results visualization
The panels show top-rule support, confidence, and lift. The summary curve tracks best lift as transactions grow from D1 to D5.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()
for ax, artifact in zip(flat_axes, artifacts):
    name, transactions, labels, rules, best = artifact
    values = [best["support"], best["confidence"], best["lift"]]
    ax.bar(["support", "confidence", "lift"], values, color=["#4c78a8", "#f58518", "#54a24b"])
    ax.set_ylim(0, max(1.5, best["lift"] + 0.5))
    ax.set_title(name.split("(")[0])
flat_axes[-1].plot(range(1, 6), [value for _, value in rows], marker="o")
flat_axes[-1].set_xticks(range(1, 6))
flat_axes[-1].set_xlabel("rung")
flat_axes[-1].set_ylabel("best lift")
flat_axes[-1].set_title("Association strength vs. complexity")
plt.tight_layout()


## Pitfall on D5: high support can still mean low lift
A common item may create a high-support rule that is less common than independence predicts. The fix is to filter rules by lift and check stability under bootstrap resampling.

In [ ]:

name, transactions, labels, k = rungs[-1]
common_item = sorted(set().union(*transactions))[0]
augmented = [set(transaction) | {"common_banner"} for transaction in transactions]
for i, transaction in enumerate(augmented):
    if i % 5 == 0 and common_item in transaction:
        transaction.remove(common_item)
wrong_s, wrong_c, wrong_l = rule_metrics(augmented, {"common_banner"}, {common_item})
filtered_rules = [rule for rule in method(augmented, min_support=0.08, min_confidence=0.1) if rule["lift"] > 1.2]
print("high-support pattern", round(wrong_s, 3), round(wrong_c, 3), round(wrong_l, 3))
print("lift-filtered rule count", len(filtered_rules))
if filtered_rules:
    print("best stable-looking rule", filtered_rules[0])


## Evaluate it + Practice
- Compare the rung metric with a no-skill baseline: random row labels, random anomaly scores, one global density, or independence-only rules.
- Run a cheap sanity check: shuffle one key input and confirm the metric or discovered artifact degrades.
- Ablate the key idea: remove scaling, held-out scoring, bandwidth selection, or lift filtering and watch the metric drop.
- Inspect failure signals: unstable assignments, high training-only fit, score thresholds that flag whole subgroups, or rules with high support but lift near 1.
- Keep the hidden labels for evaluation only; the unsupervised method must not see them during fitting.

Practice prompts:
1. Change one hyperparameter near the default and predict which rung is most sensitive.


In [ ]:
# Your experiment for practice prompt 1 goes here.

2. Add a scaling or stability check and describe the before/after metric.

In [ ]:
# Your experiment for practice prompt 2 goes here.

3. Replace the D1 toy data with your own four-to-six point example and keep the asserts auditable.

In [ ]:
# Your experiment for practice prompt 3 goes here.